# ML Experiment

**Experiment ID**: [Auto-generated]
**Hypothesis**: [TODO: What are you testing?]
**Author**: [TODO: Your name]
**Date**: [TODO: Date]

## 1. Experiment Configuration

In [ ]:
# =============================================================================
# EXPERIMENT PARAMETERS
# These can be overridden by papermill for automated runs
# =============================================================================

# Experiment identification
EXPERIMENT_NAME = 'default_experiment'
RUN_NAME = None  # Auto-generated if None

# Data parameters
DATA_PATH = 'data/raw/dataset.csv'
TARGET_COL = 'target'
TEST_SIZE = 0.2

# Model parameters
MODEL_TYPE = 'random_forest'  # Options: random_forest, xgboost, logistic
N_ESTIMATORS = 100
MAX_DEPTH = 10
LEARNING_RATE = 0.1  # For gradient boosting models

# Training parameters
SEED = 42
CV_FOLDS = 5

# Output
SAVE_MODEL = True
ARTIFACTS_DIR = 'artifacts'

## 2. Setup and Initialization

In [ ]:
# Standard library
import os
import sys
import random
import json
import logging
from pathlib import Path
from datetime import datetime

# Data handling
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ML
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Optional: XGBoost
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

# Optional: MLflow tracking
try:
    import mlflow
    import mlflow.sklearn
    HAS_MLFLOW = True
except ImportError:
    HAS_MLFLOW = False

print(f"MLflow available: {HAS_MLFLOW}")
print(f"XGBoost available: {HAS_XGB}")

In [ ]:
# =============================================================================
# REPRODUCIBILITY SETUP
# =============================================================================

def set_seeds(seed):
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # PyTorch (if available)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass

set_seeds(SEED)
print(f"Seeds set to: {SEED}")

In [ ]:
# =============================================================================
# EXPERIMENT LOGGING SETUP
# =============================================================================

# Generate experiment ID
TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
EXPERIMENT_ID = f"{EXPERIMENT_NAME}_{TIMESTAMP}"

if RUN_NAME is None:
    RUN_NAME = f"run_{TIMESTAMP}"

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Create artifacts directory
artifacts_path = Path(ARTIFACTS_DIR) / EXPERIMENT_ID
artifacts_path.mkdir(parents=True, exist_ok=True)

# Initialize experiment log
experiment_log = {
    'experiment_id': EXPERIMENT_ID,
    'run_name': RUN_NAME,
    'timestamp': TIMESTAMP,
    'parameters': {
        'data_path': DATA_PATH,
        'target_col': TARGET_COL,
        'test_size': TEST_SIZE,
        'model_type': MODEL_TYPE,
        'n_estimators': N_ESTIMATORS,
        'max_depth': MAX_DEPTH,
        'learning_rate': LEARNING_RATE,
        'seed': SEED,
        'cv_folds': CV_FOLDS,
    },
    'environment': {
        'python_version': sys.version,
        'numpy_version': np.__version__,
        'pandas_version': pd.__version__,
    },
    'metrics': {},
    'artifacts': [],
}

print(f"Experiment ID: {EXPERIMENT_ID}")
print(f"Run Name: {RUN_NAME}")
print(f"Artifacts Path: {artifacts_path}")

In [ ]:
# Optional: Initialize MLflow tracking
if HAS_MLFLOW:
    mlflow.set_experiment(EXPERIMENT_NAME)
    mlflow.start_run(run_name=RUN_NAME)
    
    # Log parameters
    mlflow.log_params(experiment_log['parameters'])
    print("MLflow run started")
else:
    print("MLflow not available, using local logging only")

## 3. Data Loading

In [ ]:
logger.info(f"Loading data from {DATA_PATH}")

df = pd.read_csv(DATA_PATH)

experiment_log['data'] = {
    'n_rows': len(df),
    'n_cols': len(df.columns),
    'columns': df.columns.tolist(),
}

print(f"Loaded {len(df):,} rows, {len(df.columns)} columns")
df.head()

## 4. Preprocessing Pipeline

In [ ]:
def preprocess_data(df, target_col):
    """Preprocess the dataset."""
    df_processed = df.copy()
    
    # TODO: Add your preprocessing steps
    # Example: Handle missing values
    # df_processed = df_processed.dropna()
    
    # Example: Encode categoricals
    categorical_cols = df_processed.select_dtypes(include=['object']).columns.tolist()
    if target_col in categorical_cols:
        categorical_cols.remove(target_col)
    
    if categorical_cols:
        df_processed = pd.get_dummies(df_processed, columns=categorical_cols, drop_first=True)
    
    return df_processed

df_processed = preprocess_data(df, TARGET_COL)
print(f"After preprocessing: {df_processed.shape}")

In [ ]:
# Prepare features and target
FEATURE_COLS = [col for col in df_processed.columns if col != TARGET_COL]

X = df_processed[FEATURE_COLS]
y = df_processed[TARGET_COL]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

experiment_log['data']['n_features'] = len(FEATURE_COLS)
experiment_log['data']['train_size'] = len(X_train)
experiment_log['data']['test_size'] = len(X_test)

print(f"Features: {len(FEATURE_COLS)}")
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

## 5. Model Definition

In [ ]:
def get_model(model_type, **kwargs):
    """Factory function to create models."""
    if model_type == 'random_forest':
        return RandomForestClassifier(
            n_estimators=kwargs.get('n_estimators', 100),
            max_depth=kwargs.get('max_depth', 10),
            random_state=kwargs.get('seed', 42),
            n_jobs=-1
        )
    elif model_type == 'logistic':
        return LogisticRegression(
            max_iter=1000,
            random_state=kwargs.get('seed', 42)
        )
    elif model_type == 'xgboost' and HAS_XGB:
        return xgb.XGBClassifier(
            n_estimators=kwargs.get('n_estimators', 100),
            max_depth=kwargs.get('max_depth', 10),
            learning_rate=kwargs.get('learning_rate', 0.1),
            random_state=kwargs.get('seed', 42),
            use_label_encoder=False,
            eval_metric='logloss'
        )
    else:
        raise ValueError(f"Unknown model type: {model_type}")

model = get_model(
    MODEL_TYPE,
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    learning_rate=LEARNING_RATE,
    seed=SEED
)

print(f"Model: {model.__class__.__name__}")

## 6. Training with Logging

In [ ]:
import time

logger.info("Starting training...")
start_time = time.time()

# Cross-validation
cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=CV_FOLDS, scoring='accuracy')

# Fit final model
model.fit(X_train_scaled, y_train)

train_time = time.time() - start_time

# Log CV metrics
experiment_log['metrics']['cv_accuracy_mean'] = float(cv_scores.mean())
experiment_log['metrics']['cv_accuracy_std'] = float(cv_scores.std())
experiment_log['metrics']['train_time_seconds'] = train_time

if HAS_MLFLOW:
    mlflow.log_metric('cv_accuracy_mean', cv_scores.mean())
    mlflow.log_metric('cv_accuracy_std', cv_scores.std())
    mlflow.log_metric('train_time_seconds', train_time)

print(f"Training completed in {train_time:.2f}s")
print(f"CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")

## 7. Evaluation with Metrics Logging

In [ ]:
# Predictions
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)

# Calculate metrics
test_metrics = {
    'test_accuracy': accuracy_score(y_test, y_pred),
    'test_precision': precision_score(y_test, y_pred, average='weighted'),
    'test_recall': recall_score(y_test, y_pred, average='weighted'),
    'test_f1': f1_score(y_test, y_pred, average='weighted'),
}

# Add ROC AUC for binary classification
if len(np.unique(y_test)) == 2:
    test_metrics['test_roc_auc'] = roc_auc_score(y_test, y_prob[:, 1])

# Log metrics
experiment_log['metrics'].update(test_metrics)

if HAS_MLFLOW:
    for name, value in test_metrics.items():
        mlflow.log_metric(name, value)

print("\nTest Metrics:")
print("=" * 40)
for name, value in test_metrics.items():
    print(f"  {name}: {value:.4f}")

In [ ]:
# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion matrix plot
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix - {EXPERIMENT_ID}')
plt.tight_layout()

# Save figure
cm_path = artifacts_path / 'confusion_matrix.png'
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
experiment_log['artifacts'].append(str(cm_path))

if HAS_MLFLOW:
    mlflow.log_artifact(str(cm_path))

plt.show()

## 8. Artifact Saving

In [ ]:
import joblib

if SAVE_MODEL:
    # Save model
    model_path = artifacts_path / 'model.joblib'
    joblib.dump(model, model_path)
    experiment_log['artifacts'].append(str(model_path))
    print(f"Model saved: {model_path}")
    
    # Save scaler
    scaler_path = artifacts_path / 'scaler.joblib'
    joblib.dump(scaler, scaler_path)
    experiment_log['artifacts'].append(str(scaler_path))
    print(f"Scaler saved: {scaler_path}")
    
    # Log to MLflow
    if HAS_MLFLOW:
        mlflow.sklearn.log_model(model, 'model')
        mlflow.log_artifact(str(scaler_path))

In [ ]:
# Save experiment log
log_path = artifacts_path / 'experiment_log.json'
with open(log_path, 'w') as f:
    json.dump(experiment_log, f, indent=2, default=str)
print(f"Experiment log saved: {log_path}")

if HAS_MLFLOW:
    mlflow.log_artifact(str(log_path))

## 9. Experiment Summary

In [ ]:
# End MLflow run
if HAS_MLFLOW:
    mlflow.end_run()
    print("MLflow run ended")

In [ ]:
# Print experiment summary
print("\n" + "=" * 60)
print("EXPERIMENT SUMMARY")
print("=" * 60)
print(f"\nExperiment ID: {EXPERIMENT_ID}")
print(f"Model: {model.__class__.__name__}")
print(f"\nKey Parameters:")
print(f"  - n_estimators: {N_ESTIMATORS}")
print(f"  - max_depth: {MAX_DEPTH}")
print(f"  - seed: {SEED}")
print(f"\nData:")
print(f"  - Train samples: {experiment_log['data']['train_size']}")
print(f"  - Test samples: {experiment_log['data']['test_size']}")
print(f"  - Features: {experiment_log['data']['n_features']}")
print(f"\nResults:")
print(f"  - CV Accuracy: {experiment_log['metrics']['cv_accuracy_mean']:.4f} (+/- {experiment_log['metrics']['cv_accuracy_std']*2:.4f})")
print(f"  - Test Accuracy: {experiment_log['metrics']['test_accuracy']:.4f}")
print(f"  - Test F1: {experiment_log['metrics']['test_f1']:.4f}")
print(f"\nArtifacts saved to: {artifacts_path}")
print("=" * 60)